# 03 · Robustness and identification diagnostics

Reads `results/` only. Interpret every row with the period bootstrap interval: the trader-clustered interval ignores common period shocks.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for cand in (start, *start.parents):
        if (cand / "run_pipeline.py").is_file() and (cand / "src").is_dir():
            return cand
    return start


ROOT = find_repo_root()
RES, FIG = ROOT / "results", ROOT / "reports" / "figures"


def load_json(name):
    p = RES / name
    if not p.exists():
        print(f"missing {p.name}: run `python run_pipeline.py` from the repo root")
        return None
    return json.loads(p.read_text())


def load_csv(name):
    p = RES / name
    if not p.exists():
        print(f"missing {p.name}: run `python run_pipeline.py` from the repo root")
        return None
    return pd.read_csv(p)


def show(name, width=800):
    p = FIG / name
    if p.exists():
        display(Image(filename=str(p), width=width))
    else:
        print(f"missing figure {name}")


manifest = load_json("manifest.json")
if manifest:
    last = manifest["runs"][manifest["latest_run"]]
    print("Latest run:", last["run_started"], "| package", last["package_version"], "| commit", last["git_commit"])


## Specification grid (24 specifications)

In [ ]:
grid = load_csv("robustness_grid.csv")
grid["sig_trader"] = grid.pvalue < 0.05
grid["sig_wild"] = grid.pvalue_week_wild < 0.05
grid.groupby(["treatment_type", "outcome"]).agg(lowest=("ate", "min"), highest=("ate", "max"),
    significant_trader=("sig_trader", "sum"), significant_wild=("sig_wild", "sum"), n=("ate", "size")).round(3)

In [ ]:
show("fig9_spec_curve.png", 1000)

## Single-change checks

In [ ]:
chk = load_csv("robustness_checks.csv")
chk[["check", "ate", "ci_lower", "ci_upper", "ci_lower_week_wild", "ci_upper_week_wild",
     "pvalue_week_wild", "n", "note"]].round(3)

## Strict exogeneity and the feedback-robust benchmark

In [ ]:
pd.Series(load_json("lead_test.json")).to_frame("value")

In [ ]:
fb = load_json("feedback_robust.json")
pd.DataFrame({k: v for k, v in fb.items() if k.startswith("instrument")}).T[["theta", "se_twoway",
    "ci_lower_twoway", "ci_upper_twoway", "first_stage_F", "n"]].round(3)

## Placebo, sensitivity, overlap

In [ ]:
load_csv("placebo.csv").round(3)[["rep", "ate", "ci_lower", "ci_upper", "pvalue_week_wild"]]

In [ ]:
sens = load_json("sensitivity.json")
print({k: sens[k] for k in ["dml_estimate", "rv_q1", "rv_q1_alpha", "rv_q1_alpha_twoway"]})
pd.DataFrame(sens["benchmarks"])[["control", "partial_r2_with_T", "partial_r2_with_Y", "adjusted_estimate",
    "sign_flips"]].round(5)

In [ ]:
ov = load_csv("overlap.csv")
print(ov.confounder.nunique(), "controls,", len(ov), "bins; bins with no treatment variation:", int((ov.sd_T == 0).sum()))
print("bins with >=90% zero treatment:", int((ov.share_T_zero >= 0.9).sum()))